# Extract Patron Data

Extracts Patron assets and their milestones, effects buffs and targets from Anno 117
into CSV and JSON files. Output goes to `results/tables/`.

Run all cells from the project root.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache

## Load assets

Setup game data for processing.

In [2]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

print(f"Total assets: {len(assets.elements)}")
print(f"Total texts: {len(assets.texts.elements)}")

Total assets: 30713
Total texts: 33147


In [3]:
asset = assets[37553] # Troop Roman Celtic Auxilia
raw_list = asset.find_value("Maintenance.Maintenances")
print("Asset 37553")
for item in raw_list:
    product = item.find_value("Product")
    print(product, product.__class__.__name__)

asset = assets[43664] # Troop Roman Murmillo Gladiator
print("Asset 43664", asset.find_value("Maintenance.Maintenances"))

Asset 37553
Denarii Asset
Wader Workforce Asset
Plebeian Workforce Asset
Equites Workforce Asset
Patrician Workforce Asset
Asset 43664 [Item (0), Item (1), Item (2), Item (3), Item (4)]


## Extract the Patron assets

Extract the patrons from `Patron` (8 assets).

In [4]:
# Misc: This allow reloading .py modules into jupyter. Only those marked with %aimport will be reloaded.
%load_ext autoreload
%autoreload 3

In [5]:
%aimport
from assetextractor.conversion.statistics.patron_extractor import PatronExtractor

extractor = PatronExtractor(assets)
# extractor_de = PatronExtractor(assets, language="german")
patrons_data = extractor.extract_all()
print(f"Processed {len(patrons_data.keys())} patrons")

Modules to reload:
all-except-skipped

Modules to skip:

Processed 8 patrons


## Print Patrons and their details

Use `extractor.print_patrons()` to see everything, or `extractor.print_patrons(guid=80562)` (Mars as example) to inspect a specific one.

In [6]:
mars_guid = 80562
mars_patron = patrons_data[mars_guid]

print(mars_patron.canonical_name)
print(mars_patron.icon.canonical_name)

patron_mars
icon_patron_mars


In [7]:
extractor.print_patrons(43594) # Ceres


             PATRON: PatronCeres (GUID: 43594)              
Shrine: Asset Pool All Ceres Shrines (GUID: 82494)
Shrines celebrate the deity and provide benefits nearby
------------------------------------------------------------
Veneration Effect: Vervactor's Plough (GUID: 43620)
All Farms can support up to 50% more field modules, allowing an increase of up to 50% productivity.
------------------------------------------------------------
Exaltation Effect: Conditor's Grace (GUID: 43603)
Increases the storage limit on all islands by 300t.
Portraits 
- Big: artwork_deity_ceres_1248_0
- Small: artwork_deity_ceres_small_88_0
Title:       Ceres Augusta
Description: Increased production of
Asset GUID:  43598
------------------------------------------------------------
Buffs: 1
  |- 1 Buff - Ceres Buff Production (GUID: 43599)
------------------------------------------------------------
Targets: 1
  |- 1 Target: Ceres Goods (GUID: 43602)
    |- 1 Target: Production Field Roman Oats (GUID: 22

## Export to JSON & Image Export

Using the `save_to_json` method from the extractor. You can pass a custom `web_base_path` to fit your webapp asset structure. If left empty, it will provide the original image game path within the asset extractor.

The `IconProcessor` is a static utility class designed for batch processing, resizing, and converting `.dds` game icons into web-ready `.webp` files.

The `IconProcessor.export_icons()` method allows for either a Flattened structure (optimized for simple galleries) or a Mirrored structure (mimicking the game's internal directory hierarchy).

This is done by using wand + magick within the IconProcessor.

In [8]:
from assetextractor.conversion.statistics.icon_processor import IconProcessor

patron_list = list(patrons_data.values())

# Define base output folder
output_root = Path("results")

# =================================================================
# SCENARIO A: MIRRORED STRUCTURE (Mimics Game Folders)
# Use this for: Webapps that need the full "base/icon_content/..." tree.
# =================================================================
print(f" Scenario A: Mirrored ".center(80, "*"))
    
# 1. Export physical files to: results/icons/base/icon_content/religion/...
IconProcessor.export_icons(
    assets=patron_list,
    output_base=output_root / "icons",
    flatten=False,      # Recreates the game's directory hierarchy
    quality=75,         # Lower quality for better compression
    resize=(128, 128)   # Resize to small icons
)

# 2. Save JSON with URLs like: "base/icon_content/religion/icon_name.webp"
extractor.save_to_json(
    file_path=output_root / "tables/patrons_en_original.json",
    web_base_path=None, # Uses relative path from the game root
    flatten=False       # Matches the mirrored export
)

# Separator
print(f"")

# =================================================================
# SCENARIO B: FLATTENED STRUCTURE (Canonical Names)
# Use this for: Simple galleries where all icons are in one folder.
# =================================================================
print(f" Scenario B: Flattened ".center(80, "*"))

# 1. Export physical files to: results/icons_flat/icon_patron_mars.webp
IconProcessor.export_icons(
    assets=patron_list,
    output_base=output_root / "icons_flat",
    flatten=True,               # All files in one folder
    use_canonical_name=True,    # Use unique canonical names
    quality=75,                 # Lower quality for better compression
    resize=(128, 128)           # Resize to small icons
)

# 2. Save JSON with URLs like: "static/assets/icon_patron_mars.webp"
extractor.save_to_json(
    file_path=output_root / "tables/patrons_en_web.json",
    web_base_path="static/assets", # Custom URL prefix
    flatten=True                   # Matches the flattened export
)

***************************** Scenario A: Mirrored *****************************
  [OK] PatronMars -> statistics\results\icons\base\icon_content\religion\icon_2d_deity_mars_0.webp
  [OK] PatronCeres -> statistics\results\icons\base\icon_content\religion\icon_2d_deity_ceres_0.webp
  [OK] PatronNeptun -> statistics\results\icons\base\icon_content\religion\icon_2d_deity_neptune_0.webp
  [OK] PatronMercury -> statistics\results\icons\base\icon_content\religion\icon_2d_deity_mercury_0.webp
  [OK] PatronEpona -> statistics\results\icons\base\icon_content\religion\icon_2d_deity_epona_0.webp
  [OK] PatronCernunnos -> statistics\results\icons\base\icon_content\religion\icon_2d_deity_cernunnos_0.webp
  [OK] PatronMinerva -> statistics\results\icons\base\icon_content\religion\icon_2d_deity_minerva_0.webp
  [OK] PatronVulcanus -> statistics\results\icons\dlc01\icon_content\religion\icon_2d_deity_vulcanus_0.webp
---
Finished! Exported: 8 | Skipped: 0
Successfully exported 8 patrons to results\table